# 00 · Setup — Ligações e Criação dos Schemas

Este notebook é o ponto de partida do pipeline ELT do Data Mart WWI (Wide World Importers).

**O que faz este notebook:**
1. Define os parâmetros de ligação às duas bases de dados
2. Testa as ligações (fonte operacional e DW)
3. Cria os schemas `bronze`, `silver` e `gold` na base de dados do DW
4. Cria as tabelas da camada `gold` (dimensões e tabelas de factos)
5. Valida a estrutura criada

---
**Arquitectura Medalão**

```
wwi (fonte OLTP)               wwi_dw (destino)
────────────────               ────────────────────────────────────
sales.*                ──E──▶  bronze.*   (cópia raw)
warehouse.*            ──T──▶  silver.*   (limpeza e transformação)
application.*          ──L──▶  gold.*     (esquema estrela / DW)
purchasing.*
```

**Modelo dimensional (Gold):**
```
                dim_date          dim_customer ──── dim_geography
                    │                  │
 dim_delivery ─── fact_sales ──── dim_product
  _method          │
               dim_employee ──── dim_geography

                dim_date          dim_customer
                    │                  │
 dim_delivery ─── fact_orders ─── dim_product
  _method          │
               dim_employee
```

## 1. Instalação de dependências

Bibliotecas necessárias:
- `psycopg2-binary` — driver PostgreSQL
- `sqlalchemy` — abstracção de ligação e execução de SQL
- `pandas` — manipulação de dados
- `python-dotenv` — carregamento de credenciais a partir do ficheiro `.env`

In [1]:
%pip install psycopg2-binary sqlalchemy pandas python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Parâmetros de ligação

As credenciais são carregadas a partir do ficheiro `.env` na mesma pasta.
Copia `.env.example` para `.env` e preenche com os teus valores antes de executar.

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()  # carrega o ficheiro .env na mesma pasta

# -- Fonte operacional (WWI OLTP) --------------------------------------------
SRC_HOST     = os.getenv("SRC_HOST")
SRC_PORT     = int(os.getenv("SRC_PORT", 5432))
SRC_DB       = os.getenv("SRC_DB")
SRC_USER     = os.getenv("SRC_USER")
SRC_PASSWORD = os.getenv("SRC_PASSWORD")

# -- Data Warehouse (destino) ------------------------------------------------
DWH_HOST     = os.getenv("DWH_HOST")
DWH_PORT     = int(os.getenv("DWH_PORT", 5432))
DWH_DB       = os.getenv("DWH_DB")
DWH_USER     = os.getenv("DWH_USER")
DWH_PASSWORD = os.getenv("DWH_PASSWORD")

# -- Schemas da arquitectura medalhão ----------------------------------------
SCHEMAS = ["bronze", "silver", "gold"]

print(f"SRC: {SRC_USER}@{SRC_HOST}:{SRC_PORT}/{SRC_DB}")
print(f"DWH: {DWH_USER}@{DWH_HOST}:{DWH_PORT}/{DWH_DB}")

SRC: dss@postgres2.ipca.pt:5432/wwi
DWH: postgres@localhost:5432/wwi_dw


## 3. Criação dos engines SQLAlchemy

In [3]:
from sqlalchemy import create_engine, text

def make_engine(host, port, db, user, password):
    url = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}"
    return create_engine(url)

engine_src = make_engine(SRC_HOST, SRC_PORT, SRC_DB, SRC_USER, SRC_PASSWORD)
engine_dwh = make_engine(DWH_HOST, DWH_PORT, DWH_DB, DWH_USER, DWH_PASSWORD)

print("Engines criados com sucesso.")

Engines criados com sucesso.


## 4. Teste das ligações

Verifica se ambas as bases de dados estão acessíveis antes de prosseguir.

In [4]:
def test_connection(engine, label):
    try:
        with engine.connect() as conn:
            result = conn.execute(text("SELECT current_database(), version()"))
            db_name, version = result.fetchone()
            print(f"✓ {label}")
            print(f"  Base de dados : {db_name}")
            print(f"  Versão        : {version.split(',')[0]}")
    except Exception as e:
        print(f"✗ {label} — ERRO: {e}")
    print()

test_connection(engine_src, f"Fonte operacional ({SRC_DB})")
test_connection(engine_dwh, f"Data Warehouse   ({DWH_DB})")

✓ Fonte operacional (wwi)
  Base de dados : wwi
  Versão        : PostgreSQL 17.9 on x86_64-pc-linux-gnu

✓ Data Warehouse   (wwi_dw)
  Base de dados : wwi_dw
  Versão        : PostgreSQL 18.3 on x86_64-windows



## 5. Criação dos schemas da arquitectura medalão

Os três schemas são criados na base de dados do DW. `IF NOT EXISTS` garante idempotência.

In [5]:
with engine_dwh.begin() as conn:
    for schema in SCHEMAS:
        conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {schema}"))
        print(f"✓ Schema '{schema}' criado (ou já existia)")

print("\nSchemas da arquitectura medalão prontos.")

✓ Schema 'bronze' criado (ou já existia)
✓ Schema 'silver' criado (ou já existia)
✓ Schema 'gold' criado (ou já existia)

Schemas da arquitectura medalão prontos.


## 6. Criação das tabelas da camada Gold

A camada gold corresponde ao Data Mart em esquema estrela (Kimball).

**Dimensões com SCD Tipo 1** (overwrite — sem histórico de versões):
- `dim_geography`, `dim_customer`, `dim_product`, `dim_employee`, `dim_delivery_method`

**Dimensão calendário** (`dim_date`): gerada programaticamente no notebook `03_gold.ipynb`.

**Outrigger**: `dim_geography` está ligada a `dim_customer` e `dim_employee` em vez de directamente às fact tables.

**Tabelas de factos**:
- `fact_sales` — granularidade: linha de factura (invoice line)
- `fact_orders` — granularidade: linha de encomenda (order line)

In [6]:
DDL_GOLD = """
-- ─────────────────────────────────────────────
-- DROP tabelas existentes (ordem inversa por FK)
-- ─────────────────────────────────────────────
DROP TABLE IF EXISTS gold.fact_orders        CASCADE;
DROP TABLE IF EXISTS gold.fact_sales         CASCADE;
DROP TABLE IF EXISTS gold.dim_customer       CASCADE;
DROP TABLE IF EXISTS gold.dim_employee       CASCADE;
DROP TABLE IF EXISTS gold.dim_product        CASCADE;
DROP TABLE IF EXISTS gold.dim_delivery_method CASCADE;
DROP TABLE IF EXISTS gold.dim_geography      CASCADE;
DROP TABLE IF EXISTS gold.dim_date           CASCADE;

-- ─────────────────────────────────────────────
-- DIMENSÕES
-- ─────────────────────────────────────────────

-- dim_date (calendário desnormalizado, gerado programaticamente)
CREATE TABLE IF NOT EXISTS gold.dim_date (
    date_key           INT          NOT NULL PRIMARY KEY,
    full_date          DATE         NOT NULL,
    day                SMALLINT     NOT NULL,
    month              SMALLINT     NOT NULL,
    month_name         VARCHAR(15)  NOT NULL,
    quarter            SMALLINT     NOT NULL,
    year               SMALLINT     NOT NULL,
    week_of_year       SMALLINT     NOT NULL,
    day_of_week        SMALLINT     NOT NULL,
    day_name           VARCHAR(10)  NOT NULL,
    is_weekend         BOOLEAN      NOT NULL,
    is_holiday         BOOLEAN      NOT NULL DEFAULT FALSE
);

-- dim_geography (outrigger — ligada via dim_customer e dim_employee)
CREATE TABLE IF NOT EXISTS gold.dim_geography (
    geography_key       SERIAL       PRIMARY KEY,
    city_name           VARCHAR(100) NOT NULL,
    state_province_name VARCHAR(100),
    country_name        VARCHAR(100)
);

-- dim_customer (SCD Tipo 2: customer_category_name, buying_group_name, is_on_credit_hold)
CREATE TABLE IF NOT EXISTS gold.dim_customer (
    customer_key           SERIAL       PRIMARY KEY,
    customer_id            INT          NOT NULL,
    customer_name          VARCHAR(100),
    customer_category_name VARCHAR(50),
    buying_group_name      VARCHAR(50),
    bill_to_customer_name  VARCHAR(100),
    phone_number           VARCHAR(100),
    is_on_credit_hold      BOOLEAN,
    geography_key          INT          REFERENCES gold.dim_geography(geography_key),
    start_date             DATE         NOT NULL DEFAULT CURRENT_DATE,
    end_date               DATE         NOT NULL DEFAULT '9999-12-31',
    is_active              BOOLEAN      NOT NULL DEFAULT TRUE
);

-- dim_product (SCD Tipo 2: brand, size, tax_rate, lead_time_days)
CREATE TABLE IF NOT EXISTS gold.dim_product (
    product_key        SERIAL         PRIMARY KEY,
    stock_item_id      INT            NOT NULL,
    stock_item_name    VARCHAR(100),
    color_name         VARCHAR(30),
    unit_package_name  VARCHAR(50),
    outer_package_name VARCHAR(50),
    brand              VARCHAR(50),
    size               VARCHAR(30),
    lead_time_days     INT,
    quantity_per_outer INT,
    is_chiller_stock   BOOLEAN,
    tax_rate           NUMERIC(5,2),
    start_date         DATE           NOT NULL DEFAULT CURRENT_DATE,
    end_date           DATE           NOT NULL DEFAULT '9999-12-31',
    is_active          BOOLEAN        NOT NULL DEFAULT TRUE
);

-- dim_employee (SCD Tipo 2: is_salesperson)
CREATE TABLE IF NOT EXISTS gold.dim_employee (
    employee_key   SERIAL       PRIMARY KEY,
    person_id      INT          NOT NULL,
    full_name      VARCHAR(100),
    preferred_name VARCHAR(50),
    is_salesperson BOOLEAN,
    geography_key  INT          REFERENCES gold.dim_geography(geography_key),
    start_date     DATE         NOT NULL DEFAULT CURRENT_DATE,
    end_date       DATE         NOT NULL DEFAULT '9999-12-31',
    is_active      BOOLEAN      NOT NULL DEFAULT TRUE
);

-- dim_delivery_method (SCD Tipo 1)
CREATE TABLE IF NOT EXISTS gold.dim_delivery_method (
    delivery_method_key  SERIAL      PRIMARY KEY,
    delivery_method_id   INT         NOT NULL UNIQUE,
    delivery_method_name VARCHAR(50)
);

-- ─────────────────────────────────────────────
-- ÍNDICES PARCIAIS (garante 1 registo corrente por business key)
-- ─────────────────────────────────────────────
CREATE UNIQUE INDEX IF NOT EXISTS ix_dim_customer_current
    ON gold.dim_customer (customer_id) WHERE is_active = TRUE;

CREATE UNIQUE INDEX IF NOT EXISTS ix_dim_product_current
    ON gold.dim_product (stock_item_id) WHERE is_active = TRUE;

CREATE UNIQUE INDEX IF NOT EXISTS ix_dim_employee_current
    ON gold.dim_employee (person_id) WHERE is_active = TRUE;

-- ─────────────────────────────────────────────
-- TABELAS DE FACTOS
-- ─────────────────────────────────────────────

-- fact_sales (granularidade: linha de factura)
CREATE TABLE IF NOT EXISTS gold.fact_sales (
    customer_key              INT          REFERENCES gold.dim_customer(customer_key),
    date_key                  INT          REFERENCES gold.dim_date(date_key),
    delivery_method_key       INT          REFERENCES gold.dim_delivery_method(delivery_method_key),
    product_key               INT          REFERENCES gold.dim_product(product_key),
    salesperson_employee_key  INT          REFERENCES gold.dim_employee(employee_key),
    invoice_id                INT,
    order_id                  INT,
    quantity                  INT,
    unit_price                NUMERIC(18,2),
    tax_amount                NUMERIC(18,2),
    line_total_excl_tax       NUMERIC(18,2)
);

-- fact_orders (granularidade: linha de encomenda)
CREATE TABLE IF NOT EXISTS gold.fact_orders (
    customer_key              INT          REFERENCES gold.dim_customer(customer_key),
    order_date_key            INT          REFERENCES gold.dim_date(date_key),
    product_key               INT          REFERENCES gold.dim_product(product_key),
    expected_delivery_date_key INT         REFERENCES gold.dim_date(date_key),
    salesperson_employee_key  INT          REFERENCES gold.dim_employee(employee_key),
    order_id                  INT,
    ordered_quantity          INT,
    picked_quantity           INT,
    backordered_quantity      INT
);

-- ─────────────────────────────────────────────
-- ÍNDICES (performance)
-- ─────────────────────────────────────────────

CREATE INDEX IF NOT EXISTS ix_fact_sales_date    ON gold.fact_sales(date_key);
CREATE INDEX IF NOT EXISTS ix_fact_sales_cust    ON gold.fact_sales(customer_key);
CREATE INDEX IF NOT EXISTS ix_fact_sales_prod    ON gold.fact_sales(product_key);
CREATE INDEX IF NOT EXISTS ix_fact_sales_emp     ON gold.fact_sales(salesperson_employee_key);
CREATE INDEX IF NOT EXISTS ix_fact_sales_dm      ON gold.fact_sales(delivery_method_key);

CREATE INDEX IF NOT EXISTS ix_fact_orders_date   ON gold.fact_orders(order_date_key);
CREATE INDEX IF NOT EXISTS ix_fact_orders_exp    ON gold.fact_orders(expected_delivery_date_key);
CREATE INDEX IF NOT EXISTS ix_fact_orders_cust   ON gold.fact_orders(customer_key);
CREATE INDEX IF NOT EXISTS ix_fact_orders_prod   ON gold.fact_orders(product_key);
CREATE INDEX IF NOT EXISTS ix_fact_orders_emp    ON gold.fact_orders(salesperson_employee_key);
"""

with engine_dwh.begin() as conn:
    conn.execute(text(DDL_GOLD))

print("✓ Tabelas da camada gold criadas com sucesso.")

✓ Tabelas da camada gold criadas com sucesso.


## 7. Validação — inventário dos schemas e tabelas criadas

In [7]:
import pandas as pd

query = """
    SELECT table_schema AS schema, table_name AS tabela
    FROM   information_schema.tables
    WHERE  table_schema IN ('bronze', 'silver', 'gold')
    ORDER  BY table_schema, table_name
"""

with engine_dwh.connect() as conn:
    df = pd.read_sql(text(query), conn)

print("Objectos criados no DW:")
print(df.to_string(index=False))

Objectos criados no DW:
schema               tabela
bronze        _load_control
bronze         buyinggroups
bronze               cities
bronze               colors
bronze            countries
bronze   customercategories
bronze            customers
bronze      deliverymethods
bronze         invoicelines
bronze             invoices
bronze           orderlines
bronze               orders
bronze         packagetypes
bronze               people
bronze       stateprovinces
bronze          stockgroups
bronze           stockitems
  gold         dim_customer
  gold             dim_date
  gold  dim_delivery_method
  gold         dim_employee
  gold        dim_geography
  gold          dim_product
  gold          fact_orders
  gold           fact_sales
silver        stg_customers
silver stg_delivery_methods
silver        stg_employees
silver        stg_geography
silver    stg_invoice_lines
silver         stg_invoices
silver      stg_order_lines
silver           stg_orders
silver         stg_produ

## 8. Validação — tabelas disponíveis na fonte WWI

In [8]:
query_src = """
    SELECT table_schema AS schema,
           table_name   AS tabela
    FROM   information_schema.tables
    WHERE  table_schema NOT IN ('pg_catalog', 'information_schema')
      AND  table_type = 'BASE TABLE'
    ORDER  BY table_schema, table_name
"""

with engine_src.connect() as conn:
    df_src = pd.read_sql(text(query_src), conn)

print(f"Tabelas disponíveis na fonte ({SRC_DB}):")
print(df_src.to_string(index=False))

Tabelas disponíveis na fonte (wwi):
schema               tabela
public         buyinggroups
public               cities
public               colors
public            countries
public   customercategories
public            customers
public customertransactions
public      deliverymethods
public         invoicelines
public             invoices
public           orderlines
public               orders
public         packagetypes
public       paymentmethods
public               people
public         specialdeals
public       stateprovinces
public          stockgroups
public           stockitems
public     transactiontypes


## 9. Resumo

Se as células anteriores correram sem erros, o ambiente está preparado:

| Passo | Estado |
|---|---|
| Ligação à fonte WWI | ✓ |
| Ligação ao DW | ✓ |
| Schema `bronze` criado | ✓ |
| Schema `silver` criado | ✓ |
| Schema `gold` criado | ✓ |
| Tabelas gold criadas | ✓ |

> **⚠ Atenção:** Verifica a célula 8 para confirmar os nomes exactos dos schemas
> e tabelas disponíveis na fonte WWI. Actualiza o ficheiro `01_bronze.ipynb`
> caso os nomes das tabelas ou schemas difiram do esperado.

**Próximo passo:** executar o notebook `01_bronze.ipynb`.